Objective: Aggregate Clickstream events using tumbling and sliding windows on event time.  Add watermarking to handle late-arriving data


In [ ]:
from pyspark.sql import SparkSession
import os

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
spark = SparkSession.builder \
    .appName('Lab3') \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", 10)

: 

In [17]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, to_timestamp
# define schema
schema = StructType([
    StructField('user_id', StringType(), True),
    StructField('page', StringType(), True),
    StructField('event_time', StringType(), True),
    StructField('revenue', StringType(), True),
])

# Step 1 — Set up streaming data with event timestamps
clicks = spark.readStream \
    .schema(schema=schema) \
    .option('maxFilesPerTrigger', 1) \
    .json('/tmp/clicks/input')

clicks = clicks.withColumn('event_ts', col=to_timestamp(col('event_time')))

In [28]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round
# Step 2 - Create a fixed tumbling window aggregation that is non overlapping counting every 5 minutes
tumbling_window = clicks.groupBy(
    window(timeColumn = col('event_ts'), windowDuration = '15 seconds').alias('ts_window')
).agg(
    count('*').alias('num_events'),
    _round(_sum(col=col('revenue')), 2).alias('total_revenue')
).orderBy(col('ts_window'))

# write the stream
q1 = tumbling_window.writeStream \
    .format(source='console') \
    .outputMode(outputMode='complete') \
    .option('truncate', False) \
    .trigger(processingTime='5 seconds') \
    .start()


26/05/26 22:33:15 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/g9/dcd31c4n74d710yhtll5jx800000gp/T/temporary-463c7dd2-d8e9-4058-9c3e-59931a5a3a23. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/26 22:33:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/26 22:33:15 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+----------+-------------+
|ts_window                                 |num_events|total_revenue|
+------------------------------------------+----------+-------------+
|{2026-05-26 22:27:15, 2026-05-26 22:27:30}|9         |2323.39      |
|{2026-05-26 22:27:30, 2026-05-26 22:27:45}|8         |1675.24      |
|{2026-05-26 22:27:45, 2026-05-26 22:28:00}|3         |408.85       |
+------------------------------------------+----------+-------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------+-------------+
|ts_window                                 |num_events|total_revenue|
+------------------------------------------+----------+-------------+
|{2026-05-26 22:27:15, 2026-05-26 22:27:30}|11        |2349.47      |
|{2026-05-26 22:27:30, 2026-05-26 22:27:45}|26        |6186.33      |
|{2026-05-26 22:27:45, 2026-05-26 22:28:00}|13        |3276.22      |
+------------------------------------------+----------+-------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------+-------------+
|ts_window                                 |num_events|total_revenue|
+------------------------------------------+----------+-------------+
|{2026-05-26 22:26:00, 2026-05-26 22:26:15}|7         |1700.15      |
|{2026-05-26 22:26:15, 2026-05-26 22:26:30}|9         |2510.39      |
|{2026-05-26 22:26:30, 2026-05-26 22:26:45}|9         |2141.79      |
|{2026-05-26 22:27:15, 2026-05-26 22:27:30}|11        |2349.47      |
|{2026-05-26 22:27:30, 2026-05-26 22:27:45}|26        |6186.33      |
|{2026-05-26 22:27:45, 2026-05-26 22:28:00}|13        |3276.22      |
+------------------------------------------+----------+-------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------+-------------+
|ts_window                                 |num_events|total_revenue|
+------------------------------------------+----------+-------------+
|{2026-05-26 22:17:45, 2026-05-26 22:18:00}|12        |2963.22      |
|{2026-05-26 22:18:00, 2026-05-26 22:18:15}|16        |3694.91      |
|{2026-05-26 22:18:15, 2026-05-26 22:18:30}|2         |526.19       |
|{2026-05-26 22:26:00, 2026-05-26 22:26:15}|7         |1700.15      |
|{2026-05-26 22:26:15, 2026-05-26 22:26:30}|9         |2510.39      |
|{2026-05-26 22:26:30, 2026-05-26 22:26:45}|9         |2141.79      |
|{2026-05-26 22:27:15, 2026-05-26 22:27:30}|11        |2349.47      |
|{2026-05-26 22:27:30, 2026-05-26 22:27:45}|26        |6186.33      |
|{2026-05-26 22:27:45, 2026-05-26 22:28:00}|13        |3276.22      |
+------------------------------------------+----------+--------

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/gdoan/code/sandbox/effective-octo-bassoon/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/gdoan/code/sandbox/effective-octo-bassoon/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [41]:
# Step 3 - Sliding window aggregation & overlapping intervals
# Count events in every 10-min window, sliding every 5 min
# Each event appears in MULTIPLE windows
sliding = clicks.groupBy(
    window(col('event_ts'), windowDuration='10 minutes', slideDuration='5 minutes').alias('sliding_window')
).agg(
    count('*').alias('num_events'),
    _round(_sum(col('revenue')), 2).alias('revenue during window')
).orderBy(
    col('sliding_window')
)

q2 = sliding.writeStream \
    .format(source='console') \
    .outputMode(outputMode='complete') \
    .trigger(processingTime='5 seconds') \
    .option('truncate', False) \
    .start()


26/05/26 22:49:10 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/g9/dcd31c4n74d710yhtll5jx800000gp/T/temporary-c04b81b3-4a12-4f92-aa0e-da13c609dd5c. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/26 22:49:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/26 22:49:10 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+----------+---------------------+
|sliding_window                            |num_events|revenue during window|
+------------------------------------------+----------+---------------------+
|{2026-05-26 22:30:00, 2026-05-26 22:40:00}|21        |4421.76              |
|{2026-05-26 22:35:00, 2026-05-26 22:45:00}|21        |4421.76              |
+------------------------------------------+----------+---------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------+---------------------+
|sliding_window                            |num_events|revenue during window|
+------------------------------------------+----------+---------------------+
|{2026-05-26 22:30:00, 2026-05-26 22:40:00}|51        |11680.54             |
|{2026-05-26 22:35:00, 2026-05-26 22:45:00}|51        |11680.54             |
+------------------------------------------+----------+---------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------+---------------------+
|sliding_window                            |num_events|revenue during window|
+------------------------------------------+----------+---------------------+
|{2026-05-26 22:30:00, 2026-05-26 22:40:00}|77        |18505.91             |
|{2026-05-26 22:35:00, 2026-05-26 22:45:00}|77        |18505.91             |
+------------------------------------------+----------+---------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------+---------------------+
|sliding_window                            |num_events|revenue during window|
+------------------------------------------+----------+---------------------+
|{2026-05-26 22:20:00, 2026-05-26 22:30:00}|30        |6791.17              |
|{2026-05-26 22:25:00, 2026-05-26 22:35:00}|30        |6791.17              |
|{2026-05-26 22:30:00, 2026-05-26 22:40:00}|77        |18505.91             |
|{2026-05-26 22:35:00, 2026-05-26 22:45:00}|77        |18505.91             |
+------------------------------------------+----------+---------------------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/gdoan/code/sandbox/effective-octo-bassoon/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/gdoan/code/sandbox/effective-octo-bassoon/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


ended


it is going to write the entire dataframe with output mode complete on. This means late data coming in will also be written. Sometimes we don't want that to happen, so we use watermarking. Watermarks are required for append only tables

In [70]:
# Step 4 - Add Watermarking
# Watermark = current max event time MINUS the threshold
# Windows older than (max_event_time - threshold) are finalized and state is dropped
#
# Key: window duration + watermark threshold must be SHORT enough that
# later batches (with newer timestamps) can push the watermark past earlier windows.
# Here: 30-second windows, 15-second watermark.
# Batch data spans ~60s back from now, so after a few batches Spark sees
# event_ts values spread 60s apart -> watermark advances -> earlier windows seal.
clicks_with_wm = clicks \
    .withWatermark(eventTime='event_ts', delayThreshold='15 seconds') \
    .groupBy(
        window(col('event_ts'), '30 seconds', '15 seconds').alias('window')
    ).agg(
        count('*').alias('num_events'),
        _round(_sum(col('revenue')), 2).alias('revenue during window')
    )

q3 = clicks_with_wm.writeStream \
    .outputMode('append') \
    .format('console') \
    .option('truncate', False) \
    .option('checkpointLocation', '/tmp/clicks/checkpoint') \
    .trigger(processingTime='5 seconds') \
    .start()


26/05/26 23:06:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/05/26 23:06:07 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+----------+---------------------+
|window|num_events|revenue during window|
+------+----------+---------------------+
+------+----------+---------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------+---------------------+
|window                                    |num_events|revenue during window|
+------------------------------------------+----------+---------------------+
|{2026-05-26 23:04:45, 2026-05-26 23:05:15}|11        |1394.27              |
|{2026-05-26 23:05:00, 2026-05-26 23:05:30}|27        |6133.33              |
+------------------------------------------+----------+---------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------+---------------------+
|window                                    |num_events|revenue during window|
+------------------------------------------+----------+---------------------+
|{2026-05-26 23:05:15, 2026-05-26 23:05:45}|33        |8570.62              |
+------------------------------------------+----------+---------------------+



26/05/27 01:25:35 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 919384 ms exceeds timeout 120000 ms
26/05/27 01:25:35 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/27 01:42:54 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [68]:
# Stop all streaming queries when done
for q in spark.streams.active:
    q.stop()
print('All streams stopped.')


All streams stopped.


26/05/26 23:05:55 WARN DAGScheduler: Failed to cancel job group ec91a1aa-76a4-4bc7-a44a-1c1a9452917d. Cannot find active jobs for it.
26/05/26 23:05:55 WARN DAGScheduler: Failed to cancel job group ec91a1aa-76a4-4bc7-a44a-1c1a9452917d. Cannot find active jobs for it.


•	Why is event time preferred over processing time for analytics workloads?
    Because network latency caused by distributed systems could misrepresent when an event actually occured as opposed to event time which is a more accurate representation of when the event actually occurred.

•	What is the difference between a tumbling and a sliding window?
    A tumbling window has no overlap between window aggregations. A sliding window can potentially overlap where an event can be captured in multiple windows

•	What problem does a watermark solve? What is the trade-off in choosing a longer vs. shorter watermark delay? 
    Watermarking solves the problem of how to handle late data as it arrives by introducing a lower bound of the minimum time we will accept given an output time.  This gap is called the watermark delay meaning after ive processed time t, i will not drop events as far back as t - watermark_delay. The tradeoff of choosing a longer watermark is that there would be more state to aggregate incurring memory and processing costs and introducing latency to compute aggregates over windows because of the increased state. However, this allows us to capture more events late as they arrive.  Shorter watermark delays save on this memory and processing latency pressure at the expense of potentially droppng more data as it arrives late.

•	When using watermarking, which output mode lets you write results only once a window is finalized — and why?
    Watermarking lets us use append as aggregates may change previous state in windows.  By introducing watermarking, we've provided a time to seal previously computed windows, creating immutable finalized aggregations for that said window